# Hearing a Fourier transform

A Fourier transform takes a signal that varies in time and reports which frequencies it contains. Spectroscopists live in both pictures: an NMR instrument records a decaying oscillation in time and the spectrum you interpret is its Fourier transform. The quickest way to build intuition is with sound, because you can listen to the time-domain signal while you look at its spectrum.

Below, three sine waves are added together. Sliders set their frequencies and how quickly the combined signal decays. The left panel is the signal in time, the right panel its spectrum, and a player lets you hear it. For playback, the frequencies are scaled up by a factor of 55 so that the default setting sounds like a chord of A, C♯ and E.

*Adapted from:* Dominic, A. J.; Cipolla, N. L.; Pfalzgraff, W. C.; Jankowski, J. A.; Rapf, R. J.; Montoya-Castillo, A. A Pedagogical Tour of the Fourier Transform with Applications to NMR and IR Spectroscopy. *J. Chem. Educ.* **2025**, *102* (5), 1972–1980. [https://doi.org/10.1021/acs.jchemed.4c01439](https://doi.org/10.1021/acs.jchemed.4c01439)

In [ ]:
%pip install -q ipympl ipywidgets

In [ ]:
%matplotlib widget

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider
from IPython.display import Audio, display

## Three waves, their sum, and its spectrum

The signal is sampled at 800 points per second for 150 seconds ($N$ = 120,000 samples). Try these:

- Set two frequencies equal and watch two peaks become one taller peak.
- Raise the decay rate. The sound dies away sooner, and the peaks in the spectrum get broader. That is exactly why NMR lines have widths: the faster the free induction decay, the broader the line.
- Set a frequency to zero. What happens to the spectrum, and to the sound?

In [ ]:
dt = 1/800                      # seconds between samples
N = 120000                      # number of samples
t = np.arange(N)*dt             # 150 s of signal
playback_rate = 44100           # samples per second for the audio player (frequencies x 55)

def wave(frequency, t):
    return np.sin(2*np.pi*frequency*t)

fig = plt.figure(figsize=(11, 4))

def update(frequency_1=FloatSlider(min=0, max=20, step=0.5, value=4),
           frequency_2=FloatSlider(min=0, max=20, step=0.5, value=5),
           frequency_3=FloatSlider(min=0, max=20, step=0.5, value=6),
           decay_rate=FloatSlider(min=0, max=0.2, step=0.01, value=0)):
    y1, y2, y3 = wave(frequency_1, t), wave(frequency_2, t), wave(frequency_3, t)
    envelope = np.exp(-decay_rate*t)
    y_sum = envelope*(y1 + y2 + y3)

    freqs = np.fft.rfftfreq(N, d=dt)
    spectrum = np.abs(np.fft.rfft(y_sum))

    fig.clear()
    ax1, ax2 = fig.subplots(1, 2)
    ax1.plot(t, y1, alpha=0.3, color='#FBB650', label=f'{frequency_1} Hz')
    ax1.plot(t, y2, alpha=0.3, color='#4E47C6', label=f'{frequency_2} Hz')
    ax1.plot(t, y3, alpha=0.3, color='#07143F', label=f'{frequency_3} Hz')
    ax1.plot(t, y_sum/3, color='black', label='sum / 3')
    ax1.set_xlim(0, 2 if decay_rate == 0 else min(150, 6/max(decay_rate, 0.04)))
    ax1.set_ylim(-1.5, 1.5)
    ax1.set_xlabel('time (s)'); ax1.set_ylabel('amplitude'); ax1.legend(loc='upper right')

    ax2.plot(freqs, spectrum, color='black')
    ax2.set_xlim(0, max(frequency_1, frequency_2, frequency_3) + 3)
    ax2.set_xlabel('frequency (Hz)'); ax2.set_ylabel('magnitude')
    fig.tight_layout()
    fig.canvas.draw_idle()

    display(Audio(y_sum, rate=playback_rate, autoplay=False))

interact(update);

## Your turn

The spectrum's peaks are narrow because the signal is long. Change `N` in the cell above to 1200 (1.5 s of signal), run the cell again, and look at the peaks. Then try 1,200,000. What does the length of the signal have to do with the sharpness of the spectrum?